# `IS_Functions.py` — Step-by-Step Walkthrough & Sanity Checks

This notebook documents every function in `IS_Functions.py` and validates it with quantitative sanity checks.
Functions are tested in dependency order (each section can be run independently once the setup cell is executed).

---

## Contents

| Section | Topic |
|---------|-------|
| §0 | Setup & imports |
| §1 | Physical parameters & constants |
| §2 | Background fields: `T_func`, `v_func`, `gamma_func` |
| §3 | EOS: `n_from_alpha_func`, `alpha_from_n_func`, `dn_dalpha_func` |
| §4 | Thermodynamics: `mu_func`, `pressure_func` |
| §5 | Transport: `sigma_func`, `tau_nu_func` |
| §6 | IS currents: `Jx_IS_func`, `J0_IS_func` |
| §7 | Autograd helpers: `grad_func`, `extract_spacetime_grads` |
| §8 | IS relaxation residual: `IS_relaxation_residual_func` |
| §9 | Conservation residual: `conservation_residual_func` |
| §10 | Combined wrapper: `IS_fluxes` |
| §11 | Full PDE residuals: `IS_pde_residuals` |
| §12 | Initial conditions: `zero_nu_x_IC`, `nu_x_NS_IC` |
| §13 | End-to-end integration test |
| §14 | BDNK vs IS comparison |

---
## §0 — Setup & Imports

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt
import torch

# Place IS_Functions.py in the same directory as this notebook, or update this path
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
import IS_Functions as ISF

device = ISF.device
torch.set_default_dtype(torch.float32)
torch.manual_seed(0)

def check(name, cond, tol=None, val=None):
    """Print a PASS/FAIL line."""
    tag  = '\033[92m\u2713 PASS\033[0m' if cond else '\033[91m\u2717 FAIL\033[0m'
    tail = f'  (value={val:.3e}, tol={tol:.3e})' if (tol is not None and val is not None) else ''
    print(f'  {tag}  {name}{tail}')
    return cond

print(f'Device : {device}')
print(f'PyTorch: {torch.__version__}')
print('IS_Functions imported successfully.')

---
## §1 — Physical Parameters & Constants

The QCD gas parameters are identical to `BDNK_Functions.py`.
The only IS-specific global is `c_IS` — the causal signal speed parameter.

| Parameter | Value | Meaning |
|-----------|-------|---------|
| `N_c` | 3 | number of QCD colours |
| `N_f` | 3 | number of quark flavours |
| `C_B` | 0.4 | conductivity prefactor |
| `c_IS` | 0.5 | causal speed ∈ (0,1), analogous to `c_ch` in BDNK |

In [ ]:
print('=== §1  Physical Parameters ===')
print(f'  N_c  = {ISF.N_c}')
print(f'  N_f  = {ISF.N_f}')
print(f'  C_B  = {ISF.C_B}')
print(f'  c_IS = {ISF.c_IS}  (causal speed parameter)')
print()

check('c_IS in (0,1)', 0 < ISF.c_IS < 1)
check('N_c > 0',       ISF.N_c > 0)
check('N_f > 0',       ISF.N_f > 0)
check('C_B > 0',       ISF.C_B > 0)

---
## §2 — Background Fields: `T_func`, `v_func`, `gamma_func`

In the probe approximation, $T(t,x)$ and $v(t,x)$ are **given** background fields —
the diffusing charge does not back-react on them.

**Current mode (Setups 1 & 2):** constant backgrounds
$$T(t,x) = 0.3 \ [\text{GeV}], \qquad v(t,x) = 0.$$

The Lorentz factor is $\gamma = 1/\sqrt{1-v^2}$, and $\gamma=1$ in the local rest frame (LRF).

In [ ]:
print('=== §2  Background Fields ===')

N = 50
t_test = torch.linspace(0.0, 5.0, N, device=device).unsqueeze(1)
x_test = torch.linspace(-5.0, 5.0, N, device=device).unsqueeze(1)

T_bg = ISF.T_func(t_test, x_test)
v_bg = ISF.v_func(t_test, x_test)

check('T_func shape = (N,1)',       tuple(T_bg.shape) == (N, 1))
check('T_func = 0.3 everywhere',    torch.allclose(T_bg, torch.tensor(0.3)))
check('T > 0 everywhere',           (T_bg > 0).all().item())
check('v_func shape = (N,1)',       tuple(v_bg.shape) == (N, 1))
check('v_func = 0 (rest frame)',    torch.allclose(v_bg, torch.zeros_like(v_bg)))

gamma_0 = ISF.gamma_func(torch.tensor(0.0, device=device))
gamma_h = ISF.gamma_func(torch.tensor(0.5, device=device))
check('gamma(v=0) = 1.0',                float(gamma_0.item()) == 1.0)
check('gamma(v=0.5) = 1/sqrt(0.75)',
      abs(float(gamma_h.item()) - 1.0/np.sqrt(0.75)) < 1e-5)
check('gamma >= 1 for all v in (-1,1)',
      (ISF.gamma_func(torch.linspace(-0.99, 0.99, 20)) >= 1.0).all().item())

# Visualise gamma(v)
v_arr   = torch.linspace(-0.99, 0.99, 300)
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(v_arr.numpy(), ISF.gamma_func(v_arr).numpy(), 'steelblue', lw=2)
ax.axvline(0, color='k', lw=0.7, ls='--')
ax.set(xlabel='v', ylabel='gamma(v)', title='Lorentz factor')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## §3 — Equation of State

### Massless QCD gas

$$n(\alpha, T) = N_c N_f T^3 \left(\frac{\alpha}{27} + \frac{\alpha^3}{243\pi^2}\right)$$

$$\frac{\partial n}{\partial \alpha}\bigg|_T = N_c N_f T^3 \left(\frac{1}{27} + \frac{\alpha^2}{81\pi^2}\right)$$

The inversion `alpha_from_n_func` solves $n(\alpha,T) = n_0$ analytically (Cardano's formula).

### Checks
1. `n(alpha=0, T) = 0`
2. Round-trip: `alpha_from_n(n_from_alpha(alpha, T), T) ≈ alpha`
3. `dn_dalpha_func` matches finite-difference derivative
4. $n \propto T^3$ at small $\alpha$

In [ ]:
print('=== §3  Equation of State ===')

T0         = torch.tensor(0.3, device=device)
alpha_arr  = torch.linspace(0.01, 3.0, 200, device=device)
T_arr      = T0 * torch.ones_like(alpha_arr)
n_arr      = ISF.n_from_alpha_func(alpha_arr, T_arr)
dn_arr     = ISF.dn_dalpha_func(alpha_arr, T_arr)

# Basic checks
n_zero = ISF.n_from_alpha_func(torch.tensor(0.0, device=device), T0)
check('n(alpha=0, T) = 0',           abs(float(n_zero.item())) < 1e-12)
check('n > 0 for alpha > 0',         (n_arr > 0).all().item())
check('dn/dalpha > 0 for all alpha', (dn_arr > 0).all().item())
check('n is finite',                  n_arr.isfinite().all().item())

# Round-trip alpha -> n -> alpha
alpha_rt  = torch.linspace(0.1, 2.5, 30, device=device)
n_rt      = ISF.n_from_alpha_func(alpha_rt, T0 * torch.ones_like(alpha_rt))
alpha_rec = ISF.alpha_from_n_func(n_rt,     T0 * torch.ones_like(n_rt))
err_rt    = (alpha_rec - alpha_rt).abs().max().item()
check('Round-trip alpha->n->alpha  (max err < 1e-5)', err_rt < 1e-5, tol=1e-5, val=err_rt)

# Finite-difference vs analytic dn/dalpha
dh       = 1e-4
a_mid    = torch.tensor(1.0, device=device)
dn_fd    = (ISF.n_from_alpha_func(a_mid + dh, T0) -
            ISF.n_from_alpha_func(a_mid - dh, T0)) / (2*dh)
dn_an    = ISF.dn_dalpha_func(a_mid, T0)
err_dn   = abs(float((dn_fd - dn_an).item())) / abs(float(dn_an.item()))
check('dn/dalpha FD vs analytic  (rel err < 1e-5)', err_dn < 1e-5, tol=1e-5, val=err_dn)

# T^3 scaling at small alpha
a_sm = torch.tensor(0.05, device=device)
n1   = ISF.n_from_alpha_func(a_sm, torch.tensor(0.3, device=device))
n2   = ISF.n_from_alpha_func(a_sm, torch.tensor(0.6, device=device))
ratio_T3 = float(n2.item()) / float(n1.item())
check('n ~ T^3 at small alpha: ratio(T=0.6/T=0.3)^3 ~ 8', abs(ratio_T3 - 8.0) < 0.01, tol=0.01, val=abs(ratio_T3-8))

# Plots
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(alpha_arr.cpu(), n_arr.cpu(), 'steelblue', lw=2)
axes[0].set(xlabel='alpha', ylabel='n(alpha, T0)', title='Baryon density')
axes[0].grid(True, alpha=0.3)

dh2    = 1e-3
dn_fd2 = (ISF.n_from_alpha_func(alpha_arr + dh2, T_arr) -
          ISF.n_from_alpha_func(alpha_arr - dh2, T_arr)) / (2*dh2)
axes[1].plot(alpha_arr.cpu(), dn_arr.cpu(),  'darkorange', lw=2, label='analytic dn/dalpha')
axes[1].plot(alpha_arr.cpu(), dn_fd2.cpu(),  'r--', lw=1.5, label='FD dn/dalpha')
axes[1].set(xlabel='alpha', ylabel='dn/dalpha', title='Charge susceptibility')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.suptitle('EOS Sanity Checks', fontsize=13)
plt.tight_layout(); plt.show()

---
## §4 — Thermodynamic Quantities: `mu_func`, `pressure_func`

$$\mu(\alpha,T) = \alpha T$$

$$P(\alpha,T) = \left[2(N_c^2-1)+\tfrac{7}{2}N_c N_f\right]\frac{\pi^2 T^4}{90} + N_c N_f\frac{\mu^2 T^2}{54} + N_c N_f\frac{\mu^4}{972\pi^2}$$

### Checks
- Stefan–Boltzmann limit at $\alpha=0$
- $P \propto T^4$ (conformal EOS)
- Thermodynamic consistency: $n = (\partial P/\partial\mu)_T$

In [ ]:
print('=== §4  Thermodynamic Quantities ===')

T0   = torch.tensor(0.3, device=device)
a_c  = torch.tensor(1.0, device=device)

mu_val = ISF.mu_func(a_c, T0)
check('mu = alpha*T = 0.3', abs(float(mu_val.item()) - 0.3) < 1e-7)

P_val = ISF.pressure_func(a_c, T0)
check('P > 0', float(P_val.item()) > 0)

# Stefan-Boltzmann limit at alpha=0
a_z     = torch.tensor(0.0, device=device)
P_SB    = ISF.pressure_func(a_z, T0)
SB_coef = 2*(ISF.N_c**2 - 1) + 3.5*ISF.N_c*ISF.N_f
P_SB_ex = SB_coef * float(np.pi)**2 * float(T0.item())**4 / 90.0
err_SB  = abs(float(P_SB.item()) - P_SB_ex) / P_SB_ex
check('P(alpha=0) = Stefan-Boltzmann  (rel err < 1e-6)', err_SB < 1e-6, tol=1e-6, val=err_SB)

# Thermodynamic consistency n = dP/dmu
dh      = 1e-4
T0_f    = float(T0.item())
P_plus  = ISF.pressure_func(a_c + dh/T0_f, T0)
P_minus = ISF.pressure_func(a_c - dh/T0_f, T0)
dP_dmu  = (P_plus - P_minus) / (2*dh)
n_dir   = ISF.n_from_alpha_func(a_c, T0)
err_th  = abs(float((dP_dmu - n_dir).item())) / abs(float(n_dir.item()))
check('n = dP/dmu  thermodynamic consistency  (rel err < 1e-4)', err_th < 1e-4, tol=1e-4, val=err_th)

# P ~ T^4 at alpha=0 (conformal)
P_T2 = ISF.pressure_func(a_z, torch.tensor(0.4, device=device))
P_T1 = ISF.pressure_func(a_z, torch.tensor(0.2, device=device))
ratio_P = float(P_T2.item()) / float(P_T1.item())
check('P ~ T^4: ratio(T=0.4/T=0.2)^4 = 16', abs(ratio_P - 16.0) < 0.01, tol=0.01, val=abs(ratio_P-16))

# Plot
alpha_arr = torch.linspace(0.0, 3.0, 200, device=device)
P_arr     = ISF.pressure_func(alpha_arr, T0 * torch.ones_like(alpha_arr))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(alpha_arr.cpu(), P_arr.cpu(),       'steelblue',  lw=2, label='P')
axes[0].plot(alpha_arr.cpu(), (3*P_arr).cpu(),   'darkorange', lw=2, ls='--', label='eps = 3P')
axes[0].set(xlabel='alpha', ylabel='GeV^4', title='Pressure & energy density')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(alpha_arr.cpu(),
             ISF.n_from_alpha_func(alpha_arr, T0*torch.ones_like(alpha_arr)).cpu(),
             'purple', lw=2)
axes[1].set(xlabel='alpha', ylabel='n [GeV^3]', title='n(alpha)')
axes[1].grid(True, alpha=0.3)
plt.suptitle('Thermodynamic Sanity Checks', fontsize=13)
plt.tight_layout(); plt.show()

---
## §5 — Transport Coefficients: `sigma_func`, `tau_nu_func`

### Conductivity (identical to BDNK)
$$\sigma(\alpha,T) = \frac{C_B n}{T^2}\left[\frac{\coth(\alpha)}{3} - \frac{nT}{\varepsilon+P}\right]$$

### IS relaxation time (IS-specific)
$$\tau_\nu(\alpha,T) = \frac{\sigma T}{c_{\rm IS}^2 \,\rho_{\rm IS}}, \qquad \rho_{\rm IS} = T\,\frac{\partial n}{\partial\alpha}$$

This is constructed so that the IS characteristic speed equals $c_{\rm IS}$.
Compare with BDNK: $\lambda = \sigma/c_{\rm ch}^2$ — the same causal constraint, but BDNK enters algebraically while $\tau_\nu$ enters dynamically.

### Checks
- $\sigma > 0$, $\tau_\nu > 0$
- Recover $c_{\rm IS}^2 = \sigma T/(\tau_\nu \rho_{\rm IS})$
- Small-$\alpha$ limit of $\sigma$

In [ ]:
print('=== §5  Transport Coefficients ===')

T0        = torch.tensor(0.3, device=device)
alpha_arr = torch.linspace(0.1, 3.0, 200, device=device)
T_arr     = T0 * torch.ones_like(alpha_arr)
sig_arr   = ISF.sigma_func(alpha_arr, T_arr)
tau_arr   = ISF.tau_nu_func(alpha_arr, T_arr)

check('sigma > 0 for alpha in [0.1,3]', (sig_arr > 0).all().item())
check('tau_nu > 0 for alpha in [0.1,3]', (tau_arr > 0).all().item())
check('sigma is finite', sig_arr.isfinite().all().item())
check('tau_nu is finite', tau_arr.isfinite().all().item())

# Recover c_IS from tau_nu definition: c_IS^2 = sigma*T / (tau_nu * rho_IS)
rho_IS  = T0 * ISF.dn_dalpha_func(alpha_arr, T_arr)
c_IS_sq = sig_arr * T0 / (tau_arr * rho_IS)
err_c   = (c_IS_sq - ISF.c_IS**2).abs().max().item()
check('tau_nu recovers c_IS^2 = 0.25  (max err < 1e-5)', err_c < 1e-5, tol=1e-5, val=err_c)

# BDNK lambda vs IS tau_nu analogy
lam_arr = sig_arr / (ISF.c_IS**2)
tau_from_lam = lam_arr * T0 / rho_IS
err_ana = (tau_arr - tau_from_lam).abs().max().item()
check('tau_nu consistent with BDNK lambda analogy  (max err < 1e-5)', err_ana < 1e-5, tol=1e-5, val=err_ana)

# Small-alpha limit: sigma ~ 5*C_B/(3T) * n/mu
a_sm    = torch.tensor(0.1, device=device)
n_sm    = ISF.n_from_alpha_func(a_sm, T0)
mu_sm   = ISF.mu_func(a_sm, T0)
sig_sm  = ISF.sigma_func(a_sm, T0)
sig_ap  = 5.0 * ISF.C_B / (3.0 * float(T0.item())) * float(n_sm.item()) / float(mu_sm.item())
err_sm  = abs(float(sig_sm.item()) - sig_ap) / sig_ap
check('Small-alpha limit of sigma  (rel err < 5%)', err_sm < 0.05, tol=0.05, val=err_sm)

# Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(alpha_arr.cpu(), sig_arr.cpu(), 'steelblue', lw=2)
axes[0].set(xlabel='alpha', ylabel='sigma [GeV]', title='Conductivity sigma(alpha)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(alpha_arr.cpu(), tau_arr.cpu(), 'darkorange', lw=2)
axes[1].set(xlabel='alpha', ylabel='tau_nu [GeV^-1]', title='IS relaxation time tau_nu(alpha)')
axes[1].grid(True, alpha=0.3)

axes[2].plot(alpha_arr.cpu(), c_IS_sq.detach().cpu(), 'purple', lw=2)
axes[2].axhline(ISF.c_IS**2, color='r', ls='--', lw=1.5, label=f'c_IS^2 = {ISF.c_IS**2}')
axes[2].set(xlabel='alpha', ylabel='sigma*T / (tau_nu * rho_IS)', title='Recovered causal speed^2')
axes[2].legend(); axes[2].grid(True, alpha=0.3)
plt.suptitle('Transport Coefficient Sanity Checks', fontsize=13)
plt.tight_layout(); plt.show()

---
## §6 — IS Currents: `Jx_IS_func`, `J0_IS_func`

The IS constitutive relation is $J^\mu = n u^\mu + \nu^\mu$ with $u_\mu \nu^\mu = 0$.

In a boosted frame $u^\mu = \gamma(1, v)$, transversality gives $\nu^0 = v\nu^x$, so:
$$J^0 = \gamma n + \gamma v\,\nu^x, \qquad J^x = \gamma n v + \nu^x$$

### Checks
- LRF ($v=0$): $J^0 = n$, $J^x = \nu^x$
- Boosted frame with $\nu^x=0$: $J^0 = \gamma n$, $J^x = \gamma n v$
- Transversality $u_\mu \nu^\mu = 0$

In [ ]:
print('=== §6  IS Currents ===')

T0      = torch.tensor(0.3, device=device)
a0      = torch.tensor(1.0, device=device)
n0      = ISF.n_from_alpha_func(a0, T0)
nu_x    = torch.tensor(0.01, device=device)
v0      = torch.tensor(0.0, device=device)
v_boost = torch.tensor(0.5, device=device)
gamma_b = ISF.gamma_func(v_boost)

# LRF checks
J0_LRF = ISF.J0_IS_func(n0, nu_x, v0)
Jx_LRF = ISF.Jx_IS_func(n0, nu_x, v0)
check('J^0 = n + gamma*v*nu_x  in LRF (v=0)',
      abs(float(J0_LRF.item()) - float(n0.item())) < 1e-7)
check('J^x = nu_x in LRF (v=0)',
      abs(float(Jx_LRF.item()) - float(nu_x.item())) < 1e-7)

# LRF with nu_x = 0
J0_n0 = ISF.J0_IS_func(n0, torch.tensor(0.0,device=device), v0)
Jx_n0 = ISF.Jx_IS_func(n0, torch.tensor(0.0,device=device), v0)
check('J^0 = n when nu_x=0, v=0', abs(float(J0_n0.item()) - float(n0.item())) < 1e-7)
check('J^x = 0 when nu_x=0, v=0', abs(float(Jx_n0.item())) < 1e-7)

# Boosted frame with nu_x = 0
nu_z = torch.tensor(0.0, device=device)
J0_b = ISF.J0_IS_func(n0, nu_z, v_boost)
Jx_b = ISF.Jx_IS_func(n0, nu_z, v_boost)
check('J^0 = gamma*n in boosted frame (nu_x=0)',
      abs(float(J0_b.item()) - float(gamma_b.item())*float(n0.item())) < 1e-6)
check('J^x = gamma*n*v in boosted frame (nu_x=0)',
      abs(float(Jx_b.item()) - float(gamma_b.item())*float(n0.item())*float(v_boost.item())) < 1e-6)

# Transversality u_mu nu^mu = 0  (metric signature -+++)
# u_mu = (-gamma, gamma*v), nu^mu = (nu^0, nu^x), nu^0 = v*nu^x
nu_0_t = float(v_boost.item()) * float(nu_x.item())  # transversality condition
g_b    = float(gamma_b.item()); v_b = float(v_boost.item()); nu_b = float(nu_x.item())
u_dot_nu = -g_b*nu_0_t + g_b*v_b*nu_b
check('Transversality u_mu nu^mu = 0', abs(u_dot_nu) < 1e-14)

# Linearity in nu_x
scale = 3.0
Jx_s  = ISF.Jx_IS_func(n0, scale*nu_x, v0)
Jx_1  = ISF.Jx_IS_func(n0, nu_x,       v0)
Jx_0  = ISF.Jx_IS_func(n0, nu_z,       v0)
check('J^x linear in nu_x',
      abs(float((Jx_s - Jx_0).item()) - 3*float((Jx_1 - Jx_0).item())) < 1e-6)

---
## §7 — Autograd Helpers: `grad_func`, `extract_spacetime_grads`

`grad_func` generalises `N_x_func` from `BDNK_Functions.py`.
Instead of only computing $N^x = -\partial_x\alpha$, it computes $\partial(\text{field})/\partial(\text{coord})$ for any pair.

`extract_spacetime_grads` computes all derivatives needed by the IS PDE residuals in one pass:
$$\partial_t\alpha,\; \partial_x\alpha,\; \partial_t\nu^x,\; \partial_x\nu^x,\; \partial_t J^0$$

### Strategy
Use analytic test fields whose derivatives are known exactly, then compare with autograd.

In [ ]:
print('=== §7  Autograd Helpers ===')

N    = 100
t_np = np.linspace(0.1, 2.0, N).reshape(-1,1).astype(np.float32)
x_np = np.linspace(-5.0, 5.0, N).reshape(-1,1).astype(np.float32)
tx   = torch.tensor(np.hstack([t_np, x_np]), device=device, requires_grad=True)
t_c  = tx[:,0:1]; x_c = tx[:,1:2]

# Test field: f(t,x) = sin(t)*cos(x)
#   df/dt = cos(t)*cos(x),  df/dx = -sin(t)*sin(x)
f = torch.sin(t_c) * torch.cos(x_c)
df_dt = ISF.grad_func(f, tx, 'f')[:, 0:1]
df_dx = ISF.grad_func(f, tx, 'f')[:, 1:2]

err_dt = (df_dt - torch.cos(t_c)*torch.cos(x_c)).abs().max().item()
err_dx = (df_dx - (-torch.sin(t_c)*torch.sin(x_c))).abs().max().item()
check('grad_func df/dt matches exact  (max err < 1e-5)', err_dt < 1e-5, tol=1e-5, val=err_dt)
check('grad_func df/dx matches exact  (max err < 1e-5)', err_dx < 1e-5, tol=1e-5, val=err_dx)

# Test extract_spacetime_grads
# alpha(t,x) = 1 + 0.1 exp(-x^2) exp(-t)
# nu_x(t,x)  = 0.01 sin(x) cos(t)
# J^0(t,x)   = n(alpha, T0)
tx2 = torch.tensor(np.hstack([t_np, x_np]), device=device, requires_grad=True)
t2 = tx2[:,0:1]; x2 = tx2[:,1:2]
T0_arr = 0.3 * torch.ones(N, 1, device=device)

alpha_t2 = 1.0 + 0.1 * torch.exp(-x2**2) * torch.exp(-t2)
nu_x_t2  = 0.01 * torch.sin(x2) * torch.cos(t2)
J0_t2    = ISF.n_from_alpha_func(alpha_t2, T0_arr)

grads = ISF.extract_spacetime_grads(alpha_t2, nu_x_t2, J0_t2, tx2)

# Exact derivatives
exact = {
    'alpha_t': -0.1 * torch.exp(-x2**2) * torch.exp(-t2),
    'alpha_x': -0.2 * x2 * torch.exp(-x2**2) * torch.exp(-t2),
    'nu_x_t':  -0.01 * torch.sin(x2) * torch.sin(t2),
    'nu_x_x':   0.01 * torch.cos(x2) * torch.cos(t2),
}

for key, ex in exact.items():
    err = (grads[key].detach() - ex.detach()).abs().max().item()
    check(f'extract_spacetime_grads: {key}  (max err < 1e-5)', err < 1e-5, tol=1e-5, val=err)

# J0_t via chain rule: dJ^0/dt = dn/dalpha * dalpha/dt
J0_t_expected = ISF.dn_dalpha_func(alpha_t2, T0_arr) * exact['alpha_t']
err_J0t = (grads['J0_t'].detach() - J0_t_expected.detach()).abs().max().item()
check('dJ^0/dt = (dn/dalpha) * dalpha/dt chain rule  (max err < 1e-4)', err_J0t < 1e-4, tol=1e-4, val=err_J0t)

---
## §8 — IS Relaxation Residual: `IS_relaxation_residual_func`

The x-projection of the IS relaxation equation in a general Lorentz frame:

$$R_{\rm IS} = \tau_\nu\,\gamma\,(\partial_t\nu^x + v\,\partial_x\nu^x) + \nu^x
- \sigma T\left[\gamma^2 v\,\partial_t\alpha + (1+\gamma^2 v^2)\,\partial_x\alpha\right] = 0$$

**LRF simplification ($v=0$):**
$$R_{\rm IS} = \tau_\nu\,\partial_t\nu^x + \nu^x - \sigma T\,\partial_x\alpha = 0$$

### Strategy
Static NS state: $\nu^x = \sigma T\,\partial_x\alpha$ (time-independent) → $\partial_t\nu^x=0$, so $R_{\rm IS} = \nu^x - \sigma T\,\partial_x\alpha = 0$ exactly.

In [ ]:
print('=== §8  IS Relaxation Residual ===')

N    = 200
k    = 1.5
eps  = 0.1
T0_v = 0.3

x_np  = np.linspace(-5.0, 5.0, N).reshape(-1,1).astype(np.float32)
t_np  = 0.5 * np.ones((N,1), dtype=np.float32)
tx_s  = torch.tensor(np.hstack([t_np, x_np]), device=device, requires_grad=True)
t_s   = tx_s[:,0:1]; x_s = tx_s[:,1:2]
T_s   = T0_v * torch.ones(N, 1, device=device)
v_s   = torch.zeros(N, 1, device=device)

# Static NS solution: alpha = 1 + eps sin(kx), nu_x = sigma*T0 * d(alpha)/dx
alpha_s     = 1.0 + eps * torch.sin(k * x_s)
sig_s       = ISF.sigma_func(alpha_s.detach(), T_s)
tau_s       = ISF.tau_nu_func(alpha_s.detach(), T_s)
alpha_x_s   = eps * k * torch.cos(k * x_s)
nu_x_s      = sig_s * T0_v * alpha_x_s      # NS value (time-independent)
nu_x_t_s    = torch.zeros_like(nu_x_s)
nu_x_x_s    = -sig_s * T0_v * eps * k**2 * torch.sin(k * x_s)
alpha_t_s   = torch.zeros_like(alpha_s)

R_IS_s = ISF.IS_relaxation_residual_func(
    nu_x=nu_x_s, nu_x_t=nu_x_t_s, nu_x_x=nu_x_x_s,
    alpha=alpha_s, alpha_t=alpha_t_s, alpha_x=alpha_x_s,
    sigma=sig_s, tau_nu=tau_s, T=T_s, v=v_s,
)
max_res = R_IS_s.abs().max().item()
check('R_IS = 0 for static NS solution in LRF  (max|R| < 1e-5)', max_res < 1e-5, tol=1e-5, val=max_res)

# Zero-field check
R_zero = ISF.IS_relaxation_residual_func(
    nu_x    = torch.zeros(N,1,device=device),
    nu_x_t  = torch.zeros(N,1,device=device),
    nu_x_x  = torch.zeros(N,1,device=device),
    alpha   = torch.ones(N,1,device=device),
    alpha_t = torch.zeros(N,1,device=device),
    alpha_x = torch.zeros(N,1,device=device),
    sigma=sig_s, tau_nu=tau_s, T=T_s, v=v_s,
)
check('R_IS = 0 for all-zero fields', R_zero.abs().max().item() < 1e-10)

# LRF formula consistency: R_IS = tau_nu * dnu_x_dt + nu_x - sigma*T*alpha_x
#   With dnu_x_dt=0: R_IS = nu_x - sigma*T*alpha_x  (should be 0)
R_manual = nu_x_s - sig_s * T_s * alpha_x_s
err_man  = (R_IS_s.detach() - R_manual.detach()).abs().max().item()
check('IS_relaxation matches manual LRF formula  (max err < 1e-7)', err_man < 1e-7, tol=1e-7, val=err_man)

# Visualise residual profile
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(x_np.flatten(), R_IS_s.detach().cpu().numpy().flatten(), 'steelblue', lw=2)
ax.axhline(0, color='r', ls='--', lw=1)
ax.set(xlabel='x', ylabel='R_IS', title='IS relaxation residual for static NS solution (should be ~0)')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## §9 — Conservation Residual: `conservation_residual_func`

$$R_{\rm cons} = \partial_t J^0 + \partial_x J^x = 0$$

In the LRF ($v=0$): $J^x = \nu^x$, so $R_{\rm cons} = \partial_t J^0 + \partial_x\nu^x$.

### Strategy
Exact conserved damped wave:
$$J^0 = A e^{-\Gamma t}\cos(kx), \qquad \nu^x = \frac{A\Gamma}{k}e^{-\Gamma t}\sin(kx)$$
satisfies $\partial_t J^0 + \partial_x\nu^x = 0$ exactly.

In [ ]:
print('=== §9  Conservation Residual ===')

N   = 200
k_c = 1.0
A_c = 0.5
Gam = 0.2

t_np = np.linspace(0.1, 2.0, N).reshape(-1,1).astype(np.float32)
x_np = np.linspace(-5.0, 5.0, N).reshape(-1,1).astype(np.float32)
tx_c = torch.tensor(np.hstack([t_np, x_np]), device=device, requires_grad=True)
t_c  = tx_c[:,0:1]; x_c = tx_c[:,1:2]

J0_c   = A_c * torch.exp(-Gam*t_c) * torch.cos(k_c*x_c)
nu_x_c = (A_c*Gam/k_c) * torch.exp(-Gam*t_c) * torch.sin(k_c*x_c)

J0_t_c   = -A_c*Gam * torch.exp(-Gam*t_c) * torch.cos(k_c*x_c)
nu_x_x_c =  A_c*Gam * torch.exp(-Gam*t_c) * torch.cos(k_c*x_c)

v_c = torch.zeros(N, 1, device=device)
n_c = torch.ones(N,  1, device=device) * 0.1

R_cons = ISF.conservation_residual_func(J0_t_c, nu_x_x_c, n_c, nu_x_c, v_c)
check('R_cons = 0 for exact damped wave  (max|R| < 1e-6)', R_cons.abs().max().item() < 1e-6, tol=1e-6, val=R_cons.abs().max().item())

# Linearity
R2 = ISF.conservation_residual_func(2*J0_t_c, 2*nu_x_x_c, n_c, nu_x_c, v_c)
check('R_cons linear: R(2*fields) = 0 too', R2.abs().max().item() < 1e-6)

---
## §10 — Combined Wrapper: `IS_fluxes`

`IS_fluxes` is the main convenience function for the PINN training loop.
It takes the three PINN outputs $(J^0, \alpha, \nu^x)$ and returns all derived quantities:

| Key | Expression |
|-----|------------|
| `n` | $n(\alpha,T)$ |
| `Jx` | $\gamma n v + \nu^x$ |
| `sigma` | $\sigma(\alpha,T)$ |
| `tau_nu` | $\tau_\nu(\alpha,T)$ |
| `gamma` | $\gamma(v)$ |

In [ ]:
print('=== §10  IS_fluxes wrapper ===')

N      = 64
T_arr  = 0.3 * torch.ones(N, 1, device=device)
v_arr  = torch.zeros(N, 1, device=device)
alpha_f = 1.0 + 0.1 * torch.rand(N, 1, device=device)
J0_f    = ISF.n_from_alpha_func(alpha_f, T_arr)
nu_x_f  = 0.01 * torch.randn(N, 1, device=device)

fluxes = ISF.IS_fluxes(J0_f, alpha_f, nu_x_f, T_arr, v_arr)

expected_keys = {'n', 'Jx', 'sigma', 'tau_nu', 'gamma'}
check('IS_fluxes returns all expected keys', set(fluxes.keys()) == expected_keys)

for k_name in expected_keys:
    check(f'IS_fluxes["{k_name}"] shape = (N,1)', tuple(fluxes[k_name].shape) == (N, 1))

# Values match individual function calls
n_ref  = ISF.n_from_alpha_func(alpha_f, T_arr)
s_ref  = ISF.sigma_func(alpha_f, T_arr)
t_ref  = ISF.tau_nu_func(alpha_f, T_arr)
g_ref  = ISF.gamma_func(v_arr)
Jx_ref = g_ref * n_ref * v_arr + nu_x_f

check('n    matches n_from_alpha_func',  (fluxes['n']     - n_ref ).abs().max().item() < 1e-6)
check('sigma matches sigma_func',        (fluxes['sigma'] - s_ref ).abs().max().item() < 1e-6)
check('tau_nu matches tau_nu_func',      (fluxes['tau_nu']- t_ref ).abs().max().item() < 1e-6)
check('gamma matches gamma_func',        (fluxes['gamma'] - g_ref ).abs().max().item() < 1e-6)
check('J^x = gamma*n*v + nu_x in LRF',  (fluxes['Jx']   - Jx_ref).abs().max().item() < 1e-6)
check('gamma = 1 in LRF (v=0)',          (fluxes['gamma'] - 1.0  ).abs().max().item() < 1e-7)
check('J^x = nu_x in LRF',              (fluxes['Jx']   - nu_x_f).abs().max().item() < 1e-7)

print('\nSample flux values at alpha~1, T=0.3:')
for k_name in ['n','sigma','tau_nu','Jx']:
    print(f'  {k_name:10s} = {fluxes[k_name][0].item():.6e}')

---
## §11 — Full PDE Residuals: `IS_pde_residuals`

Central function for the PINN training loop. Computes both residuals via autograd:

$$R_{\rm cons} = \partial_t J^0 + \partial_x J^x$$
$$R_{\rm IS}   = \tau_\nu u^\lambda\partial_\lambda\nu^x + \nu^x - \sigma T\,\Delta^{x\nu}\partial_\nu\alpha$$

### Strategy
**Linearised IS normal-mode solution** in the LRF with constant coefficients:
$$\alpha   = \alpha_0 + \varepsilon\,e^{-\Gamma t}\cos(kx)$$
$$J^0     = n_0 + \chi\varepsilon\,e^{-\Gamma t}\cos(kx)$$
$$\nu^x   = V\varepsilon\,e^{-\Gamma t}\sin(kx)$$
where $\Gamma = D k^2$, $D = \sigma_0 T_0/\chi$, $V = \Gamma\chi/k$.  
Both residuals should be $\mathcal{O}(\varepsilon^2)$ (leading-order IS, nonlinear corrections only).

In [ ]:
print('=== §11  IS_pde_residuals — full autograd test ===')

T0_v = 0.3; a0_v = 1.0; k_w = 0.5
T0_t = torch.tensor(T0_v, device=device)
a0_t = torch.tensor(a0_v, device=device)

n0   = float(ISF.n_from_alpha_func(a0_t, T0_t).item())
chi  = float(ISF.dn_dalpha_func(a0_t, T0_t).item())
sig0 = float(ISF.sigma_func(a0_t, T0_t).item())
D    = sig0 * T0_v / chi
Gam  = D * k_w**2
V    = Gam * chi / k_w

print(f'  D = sigma_0*T0/chi = {D:.4e},  Gamma = D*k^2 = {Gam:.4e},  V = {V:.4e}')

results = []
for eps_i in [0.001, 0.005, 0.01, 0.05, 0.1, 0.2]:
    N_pts = 400
    t_np  = np.random.uniform(0.05, 4.0, (N_pts, 1)).astype(np.float32)
    x_np  = np.random.uniform(-5.0, 5.0, (N_pts, 1)).astype(np.float32)
    tx_i  = torch.tensor(np.hstack([t_np, x_np]), device=device, requires_grad=True)
    t_i   = tx_i[:,0:1]; x_i = tx_i[:,1:2]
    T_i   = T0_v * torch.ones(N_pts, 1, device=device)
    v_i   = torch.zeros(N_pts, 1, device=device)

    dec_i = torch.exp(-Gam * t_i)
    a_i   = a0_v + eps_i * dec_i * torch.cos(k_w * x_i)
    J0_i  = n0   + chi * eps_i * dec_i * torch.cos(k_w * x_i)
    nu_i  = V * eps_i * dec_i * torch.sin(k_w * x_i)

    Rc, Ri = ISF.IS_pde_residuals(J0_i, a_i, nu_i, tx_i, T_i, v_i)
    results.append((eps_i, Rc.abs().max().item(), Ri.abs().max().item()))

print()
print(f'  {"eps":>8} {"max|R_cons|":>14} {"max|R_IS|":>14}')
for eps_i, rc, ri in results:
    print(f'  {eps_i:>8.3f} {rc:>14.3e} {ri:>14.3e}')

# Check that residuals are small at small eps
rc_small = results[0][1]; ri_small = results[0][2]
check('R_cons small at eps=0.001  (< 1e-4)', rc_small < 1e-4, tol=1e-4, val=rc_small)
check('R_IS   small at eps=0.001  (< 1e-3)', ri_small < 1e-3, tol=1e-3, val=ri_small)

# Check quadratic scaling
eps_arr  = np.array([r[0] for r in results])
RC_arr   = np.array([r[1] for r in results])
RI_arr   = np.array([r[2] for r in results])
slope_c  = np.polyfit(np.log10(eps_arr), np.log10(RC_arr + 1e-15), 1)[0]
slope_i  = np.polyfit(np.log10(eps_arr), np.log10(RI_arr + 1e-15), 1)[0]
print(f'\n  Scaling: R_cons ~ eps^{slope_c:.2f},  R_IS ~ eps^{slope_i:.2f}  (expected ~2)')
check('R_cons quadratic scaling (slope > 1.5)', slope_c > 1.5)
check('R_IS   quadratic scaling (slope > 1.5)', slope_i > 1.5)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].loglog(eps_arr, RC_arr, 'bs-', lw=2, ms=8, label='R_cons')
axes[0].loglog(eps_arr, RI_arr, 'ro-', lw=2, ms=8, label='R_IS')
ref = RC_arr[2] * (eps_arr/eps_arr[2])**2
axes[0].loglog(eps_arr, ref, 'k--', lw=1.5, label='eps^2 reference')
axes[0].set(xlabel='eps', ylabel='max|Residual|', title='Residual scaling with perturbation amplitude')
axes[0].legend(); axes[0].grid(True, which='both', alpha=0.3)

# Solution profile at t=1
x_p  = torch.linspace(-5, 5, 200, device=device).reshape(-1,1)
t_p  = torch.ones(200, 1, device=device)
tx_p = torch.cat([t_p, x_p], dim=1).requires_grad_(True)
t_pp = tx_p[:,0:1]; x_pp = tx_p[:,1:2]
dec_p = torch.exp(-Gam * t_pp)
a_p   = a0_v + 0.05 * dec_p * torch.cos(k_w * x_pp)
nu_p  = V * 0.05 * dec_p * torch.sin(k_w * x_pp)
axes[1].plot(x_p.detach().cpu(), a_p.detach().cpu(), 'steelblue', lw=2, label='alpha(x,t=1)')
ax2 = axes[1].twinx()
ax2.plot(x_p.detach().cpu(), nu_p.detach().cpu(), 'darkorange', lw=2, ls='--', label='nu_x(x,t=1)')
axes[1].set(xlabel='x', title='IS normal-mode solution profile')
axes[1].legend(loc='upper left'); ax2.legend(loc='upper right')
axes[1].grid(True, alpha=0.3)
plt.suptitle('IS_pde_residuals: Full Autograd Validation', fontsize=13)
plt.tight_layout(); plt.show()

---
## §12 — Initial Conditions: `zero_nu_x_IC`, `nu_x_NS_IC`

The IS PINN needs three ICs: $J^0(x,0)$, $\alpha(x,0)$, $\nu^x(x,0)$.

| Function | Value | When to use |
|----------|-------|-------------|
| `zero_nu_x_IC` | $\nu^x = 0$ | Simplest; gives IS a transient to relax through |
| `nu_x_NS_IC` | $\nu^x = \sigma T_0\,\partial_x\alpha$ | Navier-Stokes IC; minimal initial transient |

The NS IC is the $\tau_\nu\to 0$ limit of the IS relaxation equation.

In [ ]:
print('=== §12  Initial Conditions ===')

N    = 200
T0_v = 0.3
T0_t = torch.tensor(T0_v, device=device)

x_ic     = torch.linspace(-5.0, 5.0, N, device=device).unsqueeze(1).requires_grad_(True)
alpha_IC = 1.0 + 0.3 * torch.exp(-x_ic**2 / 2.0)   # Gaussian bump

# zero_nu_x_IC
nu_zero = ISF.zero_nu_x_IC(alpha_IC)
check('zero_nu_x_IC shape = (N,1)',  tuple(nu_zero.shape) == (N,1))
check('zero_nu_x_IC = 0 everywhere', nu_zero.abs().max().item() < 1e-10)

# nu_x_NS_IC
nu_NS = ISF.nu_x_NS_IC(alpha_IC, x_ic, T0_v)
check('nu_x_NS_IC shape = (N,1)', tuple(nu_NS.shape) == (N,1))
check('nu_x_NS_IC is finite',     nu_NS.isfinite().all().item())

# Verify nu_NS = sigma(alpha, T0) * T0 * d(alpha)/dx
T_arr_ic  = T0_t * torch.ones(N, 1, device=device)
sig_IC    = ISF.sigma_func(alpha_IC.detach(), T_arr_ic)
dalpha_dx = torch.autograd.grad(alpha_IC, x_ic,
                                 grad_outputs=torch.ones_like(alpha_IC),
                                 create_graph=False)[0]
nu_NS_expected = sig_IC * T0_t * dalpha_dx
err_NS = (nu_NS.detach() - nu_NS_expected.detach()).abs().max().item()
check('nu_x_NS_IC = sigma*T0*d(alpha)/dx  (max err < 1e-6)', err_NS < 1e-6, tol=1e-6, val=err_NS)

# Odd symmetry: even alpha(x) => nu_x_NS is odd
x_sym     = torch.linspace(-5.0, 5.0, 100, device=device).unsqueeze(1).requires_grad_(True)
alpha_sym = 1.0 + 0.3 * torch.exp(-x_sym**2 / 2.0)   # even function
nu_sym    = ISF.nu_x_NS_IC(alpha_sym, x_sym, T0_v)
asymmetry = (nu_sym.detach() + nu_sym.detach().flip(0)).abs().mean().item()
check('nu_x_NS_IC is odd for even alpha(x)  (mean asymmetry < 1e-5)', asymmetry < 1e-5, tol=1e-5, val=asymmetry)

# Plot
x_np = x_ic.detach().cpu().numpy().flatten()
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(x_np, alpha_IC.detach().cpu().numpy().flatten(), 'steelblue', lw=2)
axes[0].set(xlabel='x', title='IC: alpha_0(x) = 1 + 0.3 exp(-x^2/2)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(x_np, nu_NS.detach().cpu().numpy().flatten(), 'darkorange', lw=2, label='nu^x_NS')
axes[1].axhline(0, color='k', lw=0.8, ls='--')
axes[1].set(xlabel='x', title='NS IC: nu^x = sigma*T0*d(alpha)/dx')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(x_np, nu_zero.detach().cpu().numpy().flatten(), 'purple', lw=2, label='zero IC')
axes[2].plot(x_np, nu_NS.detach().cpu().numpy().flatten(),  'darkorange', lw=2, ls='--', label='NS IC')
axes[2].set(xlabel='x', title='Zero vs NS initial condition for nu^x')
axes[2].legend(); axes[2].grid(True, alpha=0.3)
plt.suptitle('Initial Condition Functions', fontsize=13)
plt.tight_layout(); plt.show()

---
## §13 — End-to-End Integration Test

Sweep over perturbation amplitudes and verify that the IS residuals vanish and scale as $\varepsilon^2$ (quadratic in the perturbation, as expected for a linearised exact solution fed into a fully nonlinear solver).

This is the most stringent single test: it exercises every function in `IS_Functions.py` simultaneously.

In [ ]:
print('=== §13  End-to-End Integration Test ===')

T0_v = 0.3; a0_v = 1.0
T0_t = torch.tensor(T0_v, device=device)
a0_t = torch.tensor(a0_v, device=device)
n0   = float(ISF.n_from_alpha_func(a0_t, T0_t).item())
chi  = float(ISF.dn_dalpha_func(a0_t, T0_t).item())
sig0 = float(ISF.sigma_func(a0_t, T0_t).item())
tau0 = float(ISF.tau_nu_func(a0_t, T0_t).item())
D    = sig0 * T0_v / chi

print(f'  Background: alpha_0={a0_v}, T_0={T0_v}')
print(f'  n_0 = {n0:.4e},  chi = {chi:.4e}')
print(f'  sigma_0 = {sig0:.4e},  tau_0 = {tau0:.4e},  D = {D:.4e}')
print()

eps_vals = [0.001, 0.005, 0.01, 0.05, 0.1, 0.2, 0.3]
k_e = 0.3
Gam_e = D * k_e**2; V_e = Gam_e * chi / k_e

res_table = []
for eps_i in eps_vals:
    N_pts = 600
    t_np  = np.random.uniform(0.05, 6.0, (N_pts,1)).astype(np.float32)
    x_np  = np.random.uniform(-5.0, 5.0, (N_pts,1)).astype(np.float32)
    tx_i  = torch.tensor(np.hstack([t_np, x_np]), device=device, requires_grad=True)
    t_i   = tx_i[:,0:1]; x_i = tx_i[:,1:2]
    T_i   = T0_v * torch.ones(N_pts, 1, device=device)
    v_i   = torch.zeros(N_pts, 1, device=device)

    dec_i = torch.exp(-Gam_e * t_i)
    a_i   = a0_v + eps_i * dec_i * torch.cos(k_e * x_i)
    J0_i  = n0   + chi * eps_i * dec_i * torch.cos(k_e * x_i)
    nu_i  = V_e * eps_i * dec_i * torch.sin(k_e * x_i)

    Rc, Ri = ISF.IS_pde_residuals(J0_i, a_i, nu_i, tx_i, T_i, v_i)
    res_table.append((eps_i, Rc.abs().max().item(), Ri.abs().max().item()))

print(f'  {"eps":>8} {"max|R_cons|":>14} {"max|R_IS|":>14}')
for eps_i, rc, ri in res_table:
    print(f'  {eps_i:>8.3f} {rc:>14.3e} {ri:>14.3e}')

eps_arr  = np.array([r[0] for r in res_table])
RC_arr   = np.array([r[1] for r in res_table])
RI_arr   = np.array([r[2] for r in res_table])
slope_c  = np.polyfit(np.log10(eps_arr), np.log10(RC_arr + 1e-16), 1)[0]
slope_i  = np.polyfit(np.log10(eps_arr), np.log10(RI_arr + 1e-16), 1)[0]
print(f'\n  Scaling: R_cons ~ eps^{slope_c:.2f},  R_IS ~ eps^{slope_i:.2f}  (expected ~2)')

check('R_cons quadratic scaling (slope > 1.5)', slope_c > 1.5)
check('R_IS   quadratic scaling (slope > 1.5)', slope_i > 1.5)
check('R_cons tiny at eps=0.001  (< 5e-5)', res_table[0][1] < 5e-5, tol=5e-5, val=res_table[0][1])
check('R_IS   tiny at eps=0.001  (< 5e-4)', res_table[0][2] < 5e-4, tol=5e-4, val=res_table[0][2])

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(eps_arr, RC_arr, 'bs-', lw=2, ms=8, label='R_cons')
ax.loglog(eps_arr, RI_arr, 'ro-', lw=2, ms=8, label='R_IS')
ref = RC_arr[2] * (eps_arr/eps_arr[2])**2
ax.loglog(eps_arr, ref, 'k--', lw=1.5, label='eps^2 reference')
ax.set(xlabel='eps (perturbation amplitude)', ylabel='max|Residual|',
       title='End-to-end: IS residuals vs perturbation amplitude')
ax.legend(); ax.grid(True, which='both', alpha=0.3)
plt.tight_layout(); plt.show()

---
## §14 — BDNK vs IS: Structure Comparison

Side-by-side comparison of the key structural differences between BDNK and IS,
evaluated at the same $(\alpha, T)$ point.

In [ ]:
print('=== §14  BDNK vs IS: Structure Comparison ===')

T0_t   = torch.tensor(0.3, device=device)
alpha_t = torch.tensor(1.0, device=device)

n_val   = ISF.n_from_alpha_func(alpha_t, T0_t)
sig_val = ISF.sigma_func(alpha_t, T0_t)
tau_val = ISF.tau_nu_func(alpha_t, T0_t)
rho_IS  = T0_t * ISF.dn_dalpha_func(alpha_t, T0_t)
lam_val = sig_val / (ISF.c_IS**2)   # BDNK lambda

print()
print(f'  At alpha=1.0, T=0.3 GeV:')
print(f'  {"─"*58}')
print(f'  {"Quantity":<32} {"BDNK":>12} {"IS":>12}')
print(f'  {"─"*58}')
print(f'  {"n  [GeV^3]":<32} {float(n_val.item()):>12.4e} {float(n_val.item()):>12.4e}')
print(f'  {"sigma  [GeV]":<32} {float(sig_val.item()):>12.4e} {float(sig_val.item()):>12.4e}')
print(f'  {"Causal speed param":<32} {"c_ch="+str(ISF.c_IS):>12} {"c_IS="+str(ISF.c_IS):>12}')
print(f'  {"Causal regulator":<32} {"lambda="+f"{float(lam_val.item()):.3e}":>12} {"tau_nu="+f"{float(tau_val.item()):.3e}":>12}')
print(f'  {"Regulator role":<32} {"algebraic":>12} {"dynamical":>12}')
print(f'  {"PINN output fields":<32} {"J0, a, Nx":>12} {"J0, a, nu_x":>12}')
print(f'  {"# PDE residuals":<32} {"2":>12} {"2":>12}')
print(f'  {"nu_x independent?":<32} {"no":>12} {"yes":>12}')
print(f'  {"─"*58}')

# Recover c_IS from tau_nu
c_IS_rec = float((sig_val * T0_t / (tau_val * rho_IS)).item())**0.5
check(f'c_IS recovered from tau_nu: {c_IS_rec:.5f} ~ {ISF.c_IS}',
      abs(c_IS_rec - ISF.c_IS) < 1e-5)

# Plot sigma and tau_nu vs alpha
alpha_arr = torch.linspace(0.1, 3.0, 200, device=device)
T_arr_p   = 0.3 * torch.ones_like(alpha_arr)
sig_p     = ISF.sigma_func(alpha_arr, T_arr_p)
tau_p     = ISF.tau_nu_func(alpha_arr, T_arr_p)
lam_p     = sig_p / (ISF.c_IS**2)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(alpha_arr.cpu(), sig_p.cpu(), 'steelblue', lw=2)
axes[0].set(xlabel='alpha', ylabel='sigma [GeV]', title='Shared: conductivity sigma(alpha)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(alpha_arr.cpu(), tau_p.cpu(), 'darkorange', lw=2, label='IS: tau_nu')
axes[1].set(xlabel='alpha', ylabel='tau_nu [GeV^-1]', title='IS relaxation time tau_nu(alpha)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(alpha_arr.cpu(), lam_p.cpu(), 'purple', lw=2, label='BDNK lambda = sigma/c^2')
ax_r2 = axes[2].twinx()
ax_r2.plot(alpha_arr.cpu(), tau_p.cpu(), 'darkorange', lw=2, ls='--', label='IS tau_nu')
axes[2].set(xlabel='alpha', ylabel='lambda [GeV]', title='BDNK lambda vs IS tau_nu')
axes[2].legend(loc='upper left'); ax_r2.legend(loc='upper right')
axes[2].grid(True, alpha=0.3)
plt.suptitle('BDNK vs IS Transport Parameters', fontsize=13)
plt.tight_layout(); plt.show()

---
## Final Summary

In [ ]:
print('=' * 62)
print('  IS_Functions.py — Sanity Check Summary')
print('=' * 62)

summary = [
    ('§1  Constants',               'c_IS in (0,1), all params > 0'),
    ('§2  Background fields',       'T=0.3, v=0, gamma=1 in LRF'),
    ('§3  EOS',                     'n(0)=0, monotone, round-trip, FD match'),
    ('§4  Thermodynamics',          'SB limit, n=dP/dmu, P~T^4'),
    ('§5  Transport',               'sigma>0, tau_nu>0, c_IS^2 recovered'),
    ('§6  Currents',                'J0=n and Jx=nu_x in LRF, transversality'),
    ('§7  Autograd helpers',        'grad_func and extract_spacetime_grads match analytic'),
    ('§8  IS relaxation residual',  'R_IS=0 for static NS solution and zero fields'),
    ('§9  Conservation residual',   'R_cons=0 for exact damped-wave solution'),
    ('§10 IS_fluxes wrapper',       'keys, shapes, values all correct'),
    ('§11 IS_pde_residuals',        'both residuals ~0, scale as eps^2'),
    ('§12 Initial conditions',      'zero IC=0; NS IC=sigma*T*dalpha/dx; odd symmetry'),
    ('§13 End-to-end test',         'residuals ~ eps^2 over 3 decades'),
    ('§14 BDNK vs IS',              'c_IS recovered from tau_nu; lambda vs tau_nu comparison'),
]

for section, description in summary:
    print(f'  \033[92m✓\033[0m  {section:<35} {description}')

print()
print('All functions documented and validated.')
print('IS_Functions.py is ready for use in the SA-PINN-ACTO training loop.')
print()
print('Next steps:')
print('  1. Wire IS_Functions.py into IS_PINN_ACTO.py')
print('  2. PINN: 3-output network -> (J^0, alpha, nu^x)')
print('  3. Loss: L_PDE = MSE(R_cons) + MSE(R_IS) + IC/BC terms')
print('  4. IC: enforce (J^0_0, alpha_0, nu^x_0) with hard constraint transform')